In [1]:
import spectre
from spectre.DataStructures.Tensor import TensorIDV2Inertial, DataVector
from spectre.IO.H5 import list_observations, open_volfiles
from spectre.IO.Exporter import ObservationId, interpolate_to_points
import os, glob, h5py
import numpy as np

In [17]:
h5files = glob.glob("*.h5")
subfile_name='VolumeData.vol'

obs_ids, obs_times = list_observations(open_volfiles(h5files, subfile_name))
def get_available_obs_ids(volume_files):
    obs_metadata = {}
    for fname in volume_files:
        with spectre.IO.H5.H5File(fname, 'r') as h5file:
            vol = h5file.get_vol('VolumeData.vol')
            for obs_id in vol.list_observation_ids():
                if obs_id not in obs_metadata:
                    npts = len(vol.get_tensor_component(obs_id, 'InertialCoordinates_x').data)
                    obs_metadata[obs_id] = npts
                    
    return sorted(obs_metadata, key=obs_metadata.get)
all_obs_ids = get_available_obs_ids([h5files[1]])

In [18]:
with h5py.File(h5files[1]) as f:
    datagrid = np.array([
            f[subfile_name][f'ObservationId{all_obs_ids[-1]}']['InertialCoordinates_x'][:],
            np.arccos(f[subfile_name][f'ObservationId{all_obs_ids[-1]}']['InertialCoordinates_y'][:]),
            f[subfile_name][f'ObservationId{all_obs_ids[-1]}']['Re(FixedSource(MMode))'][:],
            f[subfile_name][f'ObservationId{all_obs_ids[-1]}']['Im(FixedSource(MMode))'][:]
        ]).T

np.save("effsource_data",datagrid)